# Feature engineering — Base 4 — clima de Jena

Orquestra `features/temporal.py` com `BASE_4`.

Passo de `10min`. Horizonte final pendente.

## Disponibilidade

- Alvo provisório: `T (degC)`.
- Demais medições: estado em `t` para o intervalo seguinte.
- `wd (deg)` bruto: **proibido**; usar `wd_sin`/`wd_cos`.
- `-9999` deve ser ausência antes desta etapa.
- Arquivos: `catalogo_features_base4.csv` e `disponibilidade_covariaveis_base4.csv`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "config" / "projeto.yaml").is_file():
            return candidate
    raise FileNotFoundError("Execute a partir do projeto ou de uma subpasta dele.")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from series_temporais.features.regras_bases import BASE_4, validar_proibidas
from series_temporais.features.temporal import (
    build_one_step_frame,
    choque_futuro_nao_altera_features,
    chronological_train_test,
)

BASE_DIR = ROOT / "trabalho" / "bases" / "grupo4"
DATA_PATH = BASE_DIR / "base4_limpa_preparada.csv"
CATALOGO_PATH = BASE_DIR / "catalogo_features_base4.csv"
DISPONIBILIDADE_PATH = BASE_DIR / "disponibilidade_covariaveis_base4.csv"
SPEC = BASE_4
TRAIN_RATIO = 0.8


## Quadro de um passo

Cada linha é uma origem. O alvo do modelo é a variação até o passo seguinte; ele não entra nas features.

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.sort_values(SPEC.time_col, kind="stable").reset_index(drop=True)
validar_proibidas(SPEC.external_cols(), SPEC)
quadro, features = build_one_step_frame(
    df,
    time_col=SPEC.time_col,
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert len(quadro) == 414150
assert len(features) == 33
assert {"y_true", "target_delta", "target_time"}.isdisjoint(features)
quadro[features].head()


## Catálogo

O arquivo CSV fica na pasta do grupo e registra, para cada coluna de `X`, se a informação é conhecida na origem, só no passado ou antecipadamente.

In [ ]:
catalogo = SPEC.catalogo(features)
disponibilidade = SPEC.tabela_disponibilidade()
assert not catalogo.grupo.eq("revisar").any()
catalogo.to_csv(CATALOGO_PATH, index=False)
disponibilidade.to_csv(DISPONIBILIDADE_PATH, index=False)
disponibilidade


## Corte cronológico

A última linha de treino cujo alvo cruzaria a primeira origem de teste é removida.

In [ ]:
treino, teste = chronological_train_test(quadro, TRAIN_RATIO)
assert len(treino) == 331319
assert len(teste) == 82830
assert treino.target_time.max() < teste.origin_time.min()
pd.DataFrame({
    "recorte": ["treino", "teste"],
    "linhas": [len(treino), len(teste)],
    "primeira_origem": [treino.origin_time.min(), teste.origin_time.min()],
    "ultima_origem": [treino.origin_time.max(), teste.origin_time.max()],
})


## Checagem de leakage

A segunda metade da série recebe um valor artificial. As features das origens anteriores precisam permanecer idênticas.

In [ ]:
resultado = choque_futuro_nao_altera_features(
    df,
    time_col=SPEC.time_col,
    indice_inicial_do_choque=len(df) // 2,
    colunas=[SPEC.target_col, *SPEC.external_cols()],
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert resultado["iguais"] is True
assert resultado["diferenca_maxima"] == 0
resultado
